# Fase 10: Seleccion de slices informativos en CT

Este notebook estudia si la clasificacion CT puede beneficiarse de reducir cortes poco informativos. MosMedData asigna la etiqueta al estudio completo, pero el modelo 2D recibe slices individuales; por eso algunos cortes pueden aportar poca informacion para la severidad global.

La fase no sustituye el experimento CT original. Lo complementa con variantes reproducibles de seleccion de slices y mantiene el split por `study_id` para evitar fuga de datos.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

import pandas as pd

KNOWN_PROJECT_ROOT = Path('/Users/yuncaichen/Master/TFM:FINAL/TFM')

def find_project_root(start: Path) -> Path:
    candidates = [start, *start.parents, KNOWN_PROJECT_ROOT]
    for candidate in candidates:
        if (candidate / 'src' / 'config.py').exists():
            return candidate
    raise FileNotFoundError('No se encontro la raiz del proyecto con src/config.py')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / 'results' / '.matplotlib'))

import matplotlib.pyplot as plt

from src.config import config
from src.data.ct_slice_selection import SLICE_SELECTION_VARIANTS, prepare_selection_metadata

print(f'Proyecto cargado desde: {PROJECT_ROOT}')


## Preparar variantes de seleccion

Esta celda calcula caracteristicas simples por slice, como intensidad media, desviacion estandar y proporcion de pixeles por encima de umbrales. Despues crea varios CSV filtrados:

- `central30_70`: conserva cortes entre el 30% y el 70% del volumen.
- `central35_65`: conserva una zona central mas estricta.
- `top12_tissue`, `top16_tissue`, `top20_tissue`: conserva los slices con mayor cobertura de tejido por estudio.

El criterio `top-K` mantiene todos los estudios, pero reduce el numero de slices por estudio.


In [ ]:
metadata_path = config.CT_DIR / 'processed_2d_slices' / 'labels_metadata.csv'
selection_dir = config.CT_DIR / 'processed_2d_slices' / 'informative_slice_metadata'

summary_df, metadata_paths = prepare_selection_metadata(
    metadata_path=metadata_path,
    output_dir=selection_dir,
)

results_dir = PROJECT_ROOT / 'results' / 'classification' / 'ct_informative_slices'
results_dir.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(
    results_dir / 'ct_slice_selection_summary.csv',
    index=False,
)

display(summary_df.round(4))
metadata_paths


## Visualizar cuanto reduce cada variante

Aqui se representa el porcentaje de slices conservados por clase. Si una variante reduce demasiado una clase minoritaria, podria perjudicar la evaluacion. Por eso interesa ver no solo el total, sino tambien el comportamiento por etiqueta CT.


In [ ]:
plot_df = summary_df.copy()
plot_df['slice_keep_pct'] = 100 * plot_df['slice_keep_ratio']

fig, ax = plt.subplots(figsize=(10, 5), dpi=140)
for label, group in plot_df.groupby('label'):
    ax.plot(group['variant'], group['slice_keep_pct'], marker='o', label=label)
ax.set_ylabel('% de slices conservados')
ax.set_xlabel('Variante')
ax.set_title('Reduccion de slices por variante y clase CT')
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.25)
ax.legend(title='Clase')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()


## Entrenar una variante concreta

La siguiente celda esta desactivada por defecto porque entrena en modo `full` y puede tardar. Para lanzar un experimento, cambia `RUN_TRAINING = True` y ajusta `VARIANT`, `ARCHITECTURE` y `STRATEGY`.

La recomendacion inicial es `top16_tissue + resnet50 + weighted_ce`, porque ResNet50 con perdida ponderada fue el mejor enfoque CT al evaluar por estudio en la fase anterior.


In [ ]:
RUN_TRAINING = False
VARIANT = 'top16_tissue'
ARCHITECTURE = 'resnet50'
STRATEGY = 'weighted_ce'

if RUN_TRAINING:
    subprocess.run(
        [
            sys.executable,
            'scripts/run_ct_informative_slice_experiments.py',
            'train-one',
            '--variant', VARIANT,
            '--architecture', ARCHITECTURE,
            '--strategy', STRATEGY,
        ],
        check=True,
    )
else:
    print('Entrenamiento desactivado.')
    print('Para entrenar desde terminal:')
    print(f'.conda/bin/python scripts/run_ct_informative_slice_experiments.py train-one --variant {VARIANT} --architecture {ARCHITECTURE} --strategy {STRATEGY}')


## Analizar resultados entrenados

Cuando exista al menos una variante entrenada, esta celda genera tablas de comparacion por slice y por estudio. La evaluacion por estudio vuelve a agregar las probabilidades de los slices de cada `study_id`, que es la unidad original de etiqueta en MosMedData.


In [ ]:
results_dir = PROJECT_ROOT / 'results' / 'classification' / 'ct_informative_slices'
trained_summaries = sorted(results_dir.glob('*_full_summary.json'))

if trained_summaries:
    subprocess.run(
        [sys.executable, 'scripts/build_ct_informative_slice_analysis.py'],
        check=True,
    )
    metrics_path = results_dir / 'ct_informative_slice_metrics.csv'
    metrics_df = pd.read_csv(metrics_path)
    display(metrics_df.sort_values(['unit', 'f1_macro'], ascending=[True, False]).round(4))
else:
    print('Todavia no hay variantes entrenadas. Ejecuta primero una celda o comando train-one.')


## Como interpretar esta fase

Si una variante mejora F1-macro por estudio, se puede argumentar que reducir slices poco informativos ayuda a que el modelo trabaje con cortes mas representativos del volumen. Si no mejora, tambien es un resultado util: indicaria que el problema CT no se explica solo por cortes poco informativos, sino tambien por etiquetas de severidad a nivel de estudio, desbalanceo, variabilidad visual y perdida de contexto 3D.
